# Tarea: Geometría de Datos en Producción II
## Transformaciones No Lineales y Reducción de Dimensionalidad (Play Store)

**Objetivo:**
El ecosistema de aplicaciones móviles es el ejemplo perfecto de distribuciones asimétricas (el 1% de las apps se lleva el 99% de las descargas) y de exceso de dimensiones (versiones, nombres únicos, fechas). En esta tarea, aplicarán transformaciones logarítmicas para suavizar la asimetría extrema y aplicarán técnicas de reducción de dimensionalidad para purgar variables de alta cardinalidad y redundantes.

**Instrucciones:**
Resuelvan los 4 retos programando en las celdas de código. Respondan a las preguntas analíticas utilizando comentarios o celdas de texto (Markdown) debajo de cada reto.

In [11]:
import pandas as pd
import numpy as np

# Instrucción: Carga tu archivo limpio. 
# Si no lo tienes a la mano, usa el siguiente bloque para descargar y pre-limpiar lo básico:

url = "https://raw.githubusercontent.com/AndresSilva23/MineriaDeDatos/refs/heads/main/parserTensor/playstore_limpio.csv"
df = pd.read_csv(url)

# Mockup de limpieza rápida
df.drop_duplicates(subset=['App'], inplace=True)
df = df[df['Installs'] != 'Free']
df['Installs'] = df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False).astype(float)
df['Reviews'] = df['Reviews'].astype(float)

print("Datos listos. Total de apps preparadas:", len(df))

Datos listos. Total de apps preparadas: 9636


---
### Reto 1: Diagnóstico de Asimetría (El Sesgo del Mercado)
La columna `Reviews` (Cantidad de reseñas) sufre de hiper-asimetría. Hay aplicaciones con 0 reseñas y gigantes como Facebook con casi 80 millones.

1. Calcula la asimetría (`.skew()`) de la columna original `Reviews`.
2. **Pregunta de Análisis:** Escribe qué valor obtuviste. Si un valor ideal para una distribución normal de Machine Learning está entre -1 y 1, ¿qué nos dice este número sobre la forma geométrica de los datos de la Play Store?

In [12]:
# 1. Calcula y muestra el skew de 'Reviews'
asimetria_original = df['Reviews'].skew()
print("Asimetría de Reviews:", asimetria_original)


Asimetría de Reviews: 26.5422953771613


### RESPUESTA A LA PREGUNTA (Escribe tu análisis aquí):

¿Qué nos dice este número sobre la forma geométrica de los datos de la Play Store?

Un skew de 26.54 indica que la distribución de Reviews está fuertemente sesgada a la derecha: la mayoría de las apps tiene pocas reseñas, pero unos pocos outliers generan una cola larga que distorsiona toda la forma.

---
### Reto 2: La Cura Logarítmica (`np.log1p`)
Vamos a curar los gradientes del modelo comprimiendo esta variable exponencial.

1. Crea una nueva columna llamada `Reviews_Log` aplicando la transformación `np.log1p()` a la columna original `Reviews`.
2. Vuelve a calcular el `.skew()` sobre esta nueva columna.
3. **Pregunta de Análisis:** ¿Cuál es el nuevo valor de asimetría? Explica técnicamente por qué usamos `log1p` (logaritmo de $x + 1$) en lugar de un `log` tradicional en este dataset. *(Pista: Piensa en las apps que nadie ha descargado ni calificado).*

In [13]:
# 1. Aplica la transformación logarítmica
df['Reviews_Log'] = np.log1p(df['Reviews'])

# 2. Muestra la nueva asimetría
asimetria_log = df['Reviews_Log'].skew()
print("Asimetría de Reviews_Log:", asimetria_log)


Asimetría de Reviews_Log: 0.04421936534918637



### RESPUESTA A LA PREGUNTA:

¿Cuál es el nuevo valor de asimetría? Explica técnicamente por qué usamos `log1p` (logaritmo de $x + 1$) en lugar de un `log` tradicional en este dataset

El nuevo valor de asimetría es 0.044. Usamos log1p porque el dataset tiene apps con 0 reseñas, y log(0) es indefinido (-inf); log1p(0) = log(1) = 0, así que maneja los ceros sin generar errores.

---
### Reto 3: Auditoría y Purga de Alta Cardinalidad
Si le pasas a una Red Neuronal una columna donde cada fila tiene un texto distinto, la red memorizará el texto en lugar de aprender el patrón (Maldición de la Dimensionalidad).

1. Utiliza el método `.nunique()` en todo el *DataFrame* para auditar la cardinalidad.
2. Observa las columnas `App` (Nombre de la aplicación) y `Current Ver` (Versión actual como "1.0.1", "2.4", etc.).
3. Utiliza `.drop(columns=[...])` para eliminar **definitivamente** estas dos columnas de tu `df`, ya que son identificadores casi únicos que no aportan valor predictivo tabular.

In [14]:
# 1. Audita la cardinalidad
print(df.nunique())

# 2 y 3. Elimina las columnas 'App' y 'Current Ver'
df = df.drop(['App', 'Current Ver'], axis=1)

# Comprobación (Verifica que ya no existan)
print("\nColumnas restantes:", df.columns.tolist())

App               9636
Category            33
Rating              39
Reviews           5327
Size               460
Installs            20
Type                 2
Price               80
Content Rating       6
Genres             118
Last Updated      1377
Current Ver       2815
Android Ver         33
Reviews_Log       5327
dtype: int64

Columnas restantes: ['Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Android Ver', 'Reviews_Log']


---
### Reto 4: Reducción por Multicolinealidad (Correlación)
La reducción de dimensionalidad no solo elimina ruido (como los nombres), también elimina variables **redundantes**. Si dos columnas dicen matemáticamente casi lo mismo, tener ambas solo gasta RAM y causa "multicolinealidad".

1. Selecciona solo las columnas numéricas con `df.select_dtypes(include=[np.number])`.
2. Calcula la matriz de correlación con `.corr()`.
3. Observa la correlación entre `Installs` y `Reviews_Log` (o `Reviews`). 
4. **Pregunta de Análisis:** ¿Cuál es el nivel de correlación entre estas dos variables? Como arquitecto de datos, si tuvieras que reducir el dataset al mínimo tamaño posible para un modelo de regresión, ¿eliminarías una de las dos? ¿Por qué?

In [15]:
# 1 y 2. Calcula la correlación de las variables numéricas
columnas = df.select_dtypes(include=[np.number])
correlacion = columnas.corr()
print(correlacion)



               Rating   Reviews  Installs     Price  Reviews_Log
Rating       1.000000  0.055002  0.040026  0.021718     0.182502
Reviews      0.055002  1.000000  0.625150 -0.020042     0.239779
Installs     0.040026  0.625150  1.000000 -0.025462     0.263850
Price        0.021718 -0.020042 -0.025462  1.000000    -0.096608
Reviews_Log  0.182502  0.239779  0.263850 -0.096608     1.000000


### RESPUESTA A LA PREGUNTA:

¿Cuál es el nivel de correlación entre estas dos variables? Como arquitecto de datos, si tuvieras que reducir el dataset al mínimo tamaño posible para un modelo de regresión, ¿eliminarías una de las dos? ¿Por qué?

La correlación entre Installs y Reviews_Log es de 0.2638, no eliminaría ninguna de las dos, porque miden aspectos distintos del comportamiento de las apps: Installs refleja alcance, mientras que Reviews_Log refleja la opinión honesta. Una app puede tener muchas instalaciones y pocas reseñas, o al revés, así que no hay redundancia real entre ellas. Eliminar alguna le quitaría poder predictivo al modelo sin ganar nada en limpieza de datos.